In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/ennxo')
START_URLS = [
    "https://www.ennxo.com/property/for-sale",
    "https://www.ennxo.com/property/for-rent"
]
OUTPUT_CSV_FILE = BASE_DIR / 'ennxo_listing_urls.csv'

TARGET_PAGES = 3
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_optimized_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)

def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a[href^="/product/"]'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    all_urls = set()

    try:
        for start_url in START_URLS:
            logging.info(f"Starting scrape at: {start_url}")
            driver.get(start_url)
            parsed_url = urlparse(start_url)
            base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
            
            current_page = 1
            
            while current_page <= TARGET_PAGES:
                try:
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/product/"]')))
                except TimeoutException:
                    logging.warning(f"Timeout waiting for elements on page {current_page} of {start_url}. Stopping this category.")
                    break
                    
                current_links = extract_links(driver, base_url)
                all_urls.update(current_links)
                
                logging.info(f"[{start_url}] Page {current_page}: Collected {len(current_links)} URLs. Total: {len(all_urls)}")

                if current_page >= TARGET_PAGES:
                    break

                try:
                    next_button_img = driver.find_element(By.XPATH, '//img[@alt="หน้าถัดไป"]')
                    driver.execute_script("arguments[0].click();", next_button_img)
                    current_page += 1
                    time.sleep(3)
                except NoSuchElementException:
                    logging.info(f"Next page button not found on {start_url}. Reached the end.")
                    break

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    final_urls = sorted(list(all_urls))

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in final_urls:
            writer.writerow([u])

    logging.info(f"Successfully saved {len(final_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 17:02:31,459 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 17:02:34,952 - INFO - Starting scrape at: https://www.ennxo.com/property/for-sale
2026-03-29 17:02:37,657 - INFO - [https://www.ennxo.com/property/for-sale] Page 1: Collected 24 URLs. Total: 24
2026-03-29 17:02:40,818 - INFO - [https://www.ennxo.com/property/for-sale] Page 2: Collected 24 URLs. Total: 48
2026-03-29 17:02:43,903 - INFO - [https://www.ennxo.com/property/for-sale] Page 3: Collected 24 URLs. Total: 72
2026-03-29 17:02:43,904 - INFO - Starting scrape at: https://www.ennxo.com/property/for-rent
2026-03-29 17:02:47,228 - INFO - [https://www.ennxo.com/property/for-rent] Page 1: Collected 24 URLs. Total: 96
2026-03-29 17:02:50,377 - INFO - [https://www.ennxo.com/property/for-rent] Page 2: Collected 24 URLs. Total: 120
2026-03-29 17:02:53,440 - INFO - [https://www.ennxo.com/property/for-rent] Page 3: Collected 24 URLs. Total: 144
2026-0

In [2]:
import csv
import logging
import time
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/ennxo')
INPUT_CSV = BASE_DIR / 'ennxo_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'ennxo_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-extensions")
    
    return uc.Chrome(options=options, version_main=145)

def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        title = driver.find_element(By.CSS_SELECTOR, 'h1').text.strip()
        data.extend(["--- Listing Title ---", title])
    except NoSuchElementException:
        pass

    try:
        price = driver.find_element(By.XPATH, "//p[contains(text(), 'บาท')]").text.strip()
        data.extend(["\n--- Price ---", price])
    except NoSuchElementException:
        pass
        
    try:
        info_section = driver.find_elements(By.XPATH, "//h2[contains(text(), 'ข้อมูลสินค้า')]/following-sibling::div[1]/div")
        if info_section:
            data.append("\n--- Product Information ---")
            for item in info_section:
                try:
                    label = item.find_elements(By.TAG_NAME, 'p')[0].text.strip()
                    value = item.find_elements(By.TAG_NAME, 'p')[1].text.strip()
                    data.append(f"{label}: {value}")
                except Exception:
                    continue
    except NoSuchElementException:
        pass

    try:
        desc = driver.find_element(By.XPATH, "//h2[contains(text(), 'รายละเอียด')]/following-sibling::p").text.strip()
        if desc:
            data.extend(["\n--- Description ---", desc])
    except NoSuchElementException:
        pass
        
    try:
        post_date = driver.find_element(By.XPATH, "//p[contains(text(), 'โพสต์เมื่อ')]").text.strip()
        if post_date:
            data.extend(["\n--- Post Date ---", post_date])
    except NoSuchElementException:
        pass

    return "\n".join(data)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    driver.get(url)
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2026-03-29 17:04:22,587 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 17:04:42,265 - INFO - Progress: [10/144] scraped.
2026-03-29 17:04:57,146 - INFO - Progress: [20/144] scraped.
2026-03-29 17:05:13,137 - INFO - Progress: [30/144] scraped.
2026-03-29 17:05:27,582 - INFO - Progress: [40/144] scraped.
2026-03-29 17:05:49,721 - INFO - Progress: [50/144] scraped.
2026-03-29 17:06:05,052 - INFO - Progress: [60/144] scraped.
2026-03-29 17:06:21,072 - INFO - Progress: [70/144] scraped.
2026-03-29 17:06:38,122 - INFO - Progress: [80/144] scraped.
2026-03-29 17:06:51,914 - INFO - Progress: [90/144] scraped.
2026-03-29 17:07:14,466 - INFO - Progress: [100/144] scraped.
2026-03-29 17:07:30,392 - INFO - Progress: [110/144] scraped.
2026-03-29 17:07:47,793 - INFO - Progress: [120/144] scraped.
2026-03-29 17:08:02,288 - INFO - Progress: [130/144] scraped.
2026-03-29 17:08:19,205 - INFO - Progress: [140/144] scraped.
2026-03